# 03 — LLM Supervised Fine-Tuning (Trendyol-LLM-7B, QLoRA, Unsloth)

This notebook performs citation-aware QLoRA fine-tuning of `Trendyol/Trendyol-LLM-7B-chat-v4.1.0` for Turkish legal question answering. The resulting LoRA adapter is the generator used in the `A5` (full fine-tune) ablation cell.

**Framework choice.** Earlier iterations on the bare `TRL + transformers + peft + bitsandbytes` stack surfaced a chain of mixed-precision and gradient-checkpointing issues that were brittle to library versions. The notebook standardizes on **Unsloth's `FastLanguageModel`**, which encapsulates a known-good T4 / Qwen2 / 4-bit / LoRA configuration (dtype, gradient checkpointing, scaler, response-masking) and runs roughly 2× faster than a hand-tuned setup.

**Approach.**
- 4-bit dynamic quantization (Unsloth default) on the base model.
- LoRA `r=16, α=32` on `q / k / v / o / gate / up / down` projections.
- `max_seq_length=2048`, pre-filter examples beyond 1,950 tokens to leave a safety margin.
- Completion-only loss masking via `unsloth.chat_templates.train_on_responses_only` (the newer TRL `DataCollatorForCompletionOnlyLM` does not compose cleanly with Unsloth's `remove_unused_columns=False`).
- AdamW 8-bit optimizer, cosine schedule, `lr=2e-4`, 2 epochs, `batch=1 × grad_accum=16`.

**System prompt.** Enforces (1) grounding strictly in the provided article, (2) `[KısaAd m.MaddeNo]` citation format, (3) explicit refusal when no answer is supported.

**Hardware.** Kaggle T4×1. Wall time ~3–4 hours.

**Validation strategy.** Periodic in-trainer evaluation is disabled because the response-only collator's column handling conflicts with Unsloth's wrapper under TRL ≥ 0.21. Three pre-training sanity guards (label-mask ratio, no-grad forward, grad-mode forward + gradient-norm) catch silent-loss issues before the long run. End-to-end evaluation is handled in `04_ablation_eval.ipynb` and `src/eval/run_eval.py`.

**Output.** `Trendyol-LLM-7B-chat-v4.1.0-tr-legal-qlora/` — LoRA adapter (~200 MB).

In [ ]:
%%capture
# Unsloth kurulum (Kaggle T4'te ~2-3 dk)
# Robust kurulum: torch'u Kaggle'in onceden yukledigi versiyonda birak, sadece unsloth + uyumlu trl ekle
!pip install -q --upgrade pip
!pip install -q --no-deps bitsandbytes accelerate xformers peft "trl<0.12.0" triton cut_cross_entropy unsloth_zoo
!pip install -q sentencepiece protobuf "datasets>=2.16.0" huggingface_hub hf_transfer
# transformers 5.0 Unsloth+TRL 0.12 ile uyumsuz (tokenizer kwarg dropped, fix_untrained_tokens None alir). 4.x'e pinle.
!pip install -q "transformers>=4.46,<5.0"
!pip install -q --no-deps unsloth

In [ ]:
import os
os.environ['WANDB_DISABLED'] = 'true'

import json, torch
from pathlib import Path
from datasets import Dataset

MODEL_NAME = 'Trendyol/Trendyol-LLM-7B-chat-v4.1.0'
OUTPUT_DIR = Path(f'/kaggle/working/{MODEL_NAME.split("/")[-1]}-tr-legal-qlora')

MAX_SEQ_LEN = 2048
PREFILTER_TOK_MAX = 1950

DATASET_CANDIDATES = [
    Path('/kaggle/input/turkish-legal-rag-finetune'),
    Path('/kaggle/input/datasets/hasanemreusta/turkish-legal-rag-finetune'),
]
INPUT_DIR = next((p for p in DATASET_CANDIDATES if p.exists()), DATASET_CANDIDATES[0])
SYNTHETIC_QA = INPUT_DIR / 'synthetic_qa.jsonl'

print('CUDA:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('Dataset:', SYNTHETIC_QA, 'exists:', SYNTHETIC_QA.exists())

## Model loading with Unsloth

`FastLanguageModel` performs the 4-bit load and the LoRA wrap in one call. Trendyol-LLM-7B uses Qwen2 architecture, which Unsloth supports natively.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,                     # Unsloth otomatik secer (T4 -> fp16, A100 -> bf16)
    load_in_4bit=True,
    trust_remote_code=True,
)

# LoRA ekle
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=32,
    lora_dropout=0,                         # Unsloth fast LoRA kernels icin 0 olmali (yoksa fast path kapanir, ~2x yavas)
    bias='none',
    use_gradient_checkpointing='unsloth',   # Unsloth'un kendi GC implementasyonu (bug-free)
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

print('Model + LoRA hazir.')
model.print_trainable_parameters()

## Data preparation and pre-filtering

Applies the chat template (system + user + assistant turns) and pre-filters examples whose tokenized length exceeds 1,950 (leaves headroom under the 2,048 `max_seq_length`). The newer TRL `SFTTrainer` no longer lazy-tokenizes from `dataset_text_field`, so we tokenize explicitly and drop the raw `text` column.

In [ ]:
SYSTEM = """Sen Türk hukuku alanında uzman bir asistansın. Sana sorulan soruyu,
SADECE verilen kanun maddelerine dayanarak cevapla.

Kurallar:
1. Cevap mutlaka verilen maddelerden çıkarılabilir olmalı; uydurma bilgi verme.
2. Her iddianın sonunda atıf yap: \"[KısaAd m.MaddeNo]\" formatında.
3. Eğer maddelerde cevap yoksa \"Verilen kaynaklarda bu sorunun cevabı bulunmamaktadır.\" de.
4. Net, kısa ve hukuki üslupta cevap ver."""

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def make_example(r):
    user = f"Soru: {r['question']}\n\nKaynak Madde:\n[{r['kanun_short']} m.{r['madde_no']}]\n{r['madde_text'][:1500]}\n\nCevap:"
    text = tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM},
         {'role': 'user', 'content': user},
         {'role': 'assistant', 'content': r['answer']}],
        tokenize=False,
    )
    return {'text': text}

raw_rows = [make_example(json.loads(l)) for l in SYNTHETIC_QA.open(encoding='utf-8')]
before = len(raw_rows)

def tok_len(r):
    return len(tokenizer.encode(r['text']))

rows = [r for r in raw_rows if tok_len(r) <= PREFILTER_TOK_MAX]
after = len(rows)
print(f'Pre-filter: {before} -> {after} ornek ({before-after} atildi)')

all_lens = sorted(tok_len(r) for r in rows)
n = len(all_lens)
print(f'Token uzunlugu: min={all_lens[0]} p50={all_lens[n//2]} p95={all_lens[int(n*0.95)]} max={all_lens[-1]}')
assert all_lens[-1] <= MAX_SEQ_LEN

print('\n--- Ornek (son 200 char) ---')
print(rows[0]['text'][-200:])

ds = Dataset.from_list(rows).train_test_split(test_size=0.05, seed=42)
print('Raw:', ds)

# Yeni TRL'de SFTTrainer raw text'i lazy tokenize etmiyor; pre-tokenize ediyoruz, text kolonunu kaldiriyoruz.
def _tokenize(ex):
    return tokenizer(ex['text'], truncation=True, max_length=MAX_SEQ_LEN, padding=False)

ds = ds.map(_tokenize, batched=True, remove_columns=['text'], desc='tokenize')
print('Tokenized:', ds)
print('Ornek input_ids len:', len(ds['train'][0]['input_ids']))

## Trainer construction (TRL `SFTTrainer` + Unsloth)

Completion-only masking is applied via `unsloth.chat_templates.train_on_responses_only` after `SFTTrainer` construction. Periodic eval is disabled because TRL ≥ 0.21 + Unsloth's wrapper leaks the `text` column into the collator and breaks evaluation.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

# Unsloth dtype'a uygun trainer ayari
is_bf16 = torch.cuda.is_bf16_supported()
print(f'BF16 supported: {is_bf16}')

sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    logging_steps=10,
    eval_strategy='no',                # Periodic eval kapali (TRL+transformers 4.57 yeni API ile text kolon collator'a sizmasi sorunu)
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    bf16=is_bf16,
    fp16=not is_bf16,
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    optim='adamw_8bit',
    seed=42,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=sft_config,
    train_dataset=ds['train'],
)

# Unsloth native completion-only masking: sadece assistant turn'undeki tokenlar label, geri kalan -100
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)

print('--- Trainer hazir ---')
print('Train size:', len(ds['train']))

## Sanity guards — forward and gradient flow

Three checks executed once before the long run:

1. **Label mask ratio** — confirms that response-only masking leaves a non-trivial fraction of tokens contributing to the loss (≥5%).
2. **No-grad forward** — verifies that a clean forward pass produces a finite loss.
3. **Grad-mode forward + backward** — runs a full step and confirms a non-zero gradient norm reaches the trainable parameters.

In [ ]:
# Guard 1: masking ratio
dl = trainer.get_train_dataloader()
ratios = []
first_batch = None
for i, b in enumerate(dl):
    if first_batch is None:
        first_batch = b
    valid = (b['labels'] != -100).sum().item()
    total = b['labels'].numel()
    ratios.append(valid / total)
    if i >= 2:
        break
avg_ratio = sum(ratios) / len(ratios)
print(f'Label ratios: {[f"{r:.2%}" for r in ratios]} | avg={avg_ratio:.2%}')
if avg_ratio < 0.05:
    raise ValueError(f'Masking sorunu: avg ratio {avg_ratio:.2%}')

# Guard 2: no-grad forward
with torch.no_grad():
    batch_dev = {k: v.to(trainer.model.device) for k, v in first_batch.items()}
    out = trainer.model(**batch_dev)
print(f'No-grad forward loss: {out.loss.item():.4f}')
if not torch.isfinite(out.loss):
    raise ValueError(f'No-grad loss not finite: {out.loss.item()}')

# Guard 3: grad-mode forward + gradient flow
trainer.model.train()
for p in trainer.model.parameters():
    if p.grad is not None:
        p.grad = None

out = trainer.model(**batch_dev)
print(f'Grad-mode forward loss: {out.loss.item():.4f}')
if not torch.isfinite(out.loss):
    raise ValueError(f'Grad-mode loss not finite: {out.loss.item()}')

out.loss.backward()
total_norm = 0.0
n_with_grad = 0
for p in trainer.model.parameters():
    if p.requires_grad and p.grad is not None:
        g = p.grad.detach().float().norm().item()
        total_norm += g * g
        if g > 0:
            n_with_grad += 1
total_norm = total_norm ** 0.5
print(f'Params with grad: {n_with_grad} | total grad norm: {total_norm:.4f}')
if total_norm == 0.0:
    raise ValueError('Gradient akmiyor!')

for p in trainer.model.parameters():
    if p.grad is not None:
        p.grad = None
trainer.model.zero_grad()

print('OK Sanity gecti.')

## Smoke training (30 steps on 200 examples, ~8 minutes)

A short dry run on a subset to confirm the loss curve trends downward in the expected range (~1.5 → ~0.4) before committing to the full run.

In [ ]:
RUN_SMOKE = True

if RUN_SMOKE:
    smoke_ds = ds['train'].select(range(min(200, len(ds['train']))))
    smoke_cfg = SFTConfig(
        output_dir=str(OUTPUT_DIR) + '_smoke',
        max_steps=30,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        lr_scheduler_type='cosine',
        logging_steps=5,
        eval_strategy='no',
        save_strategy='no',
        bf16=is_bf16,
        fp16=not is_bf16,
        max_seq_length=MAX_SEQ_LEN,
        packing=False,
            optim='adamw_8bit',
        seed=42,
        report_to='none',
    )
    smoke_trainer = SFTTrainer(
        model=trainer.model,
        tokenizer=tokenizer,
        args=smoke_cfg,
        train_dataset=smoke_ds,
    )
    smoke_trainer = train_on_responses_only(
        smoke_trainer,
        instruction_part='<|im_start|>user\n',
        response_part='<|im_start|>assistant\n',
    )
    print('Smoke training (30 step) basliyor...')
    smoke_trainer.train()
    print("\nSmoke OK ise (loss monoton dustu, sayisal degerler 1.5-0.4 araliginda) full hucresine gec.")
else:
    print('Smoke atlandi.')

## Full training run

~3–4 hours on T4×1.

In [ ]:
trainer.train()
trainer.save_model(str(OUTPUT_DIR))
print('Done:', OUTPUT_DIR)

## Post-training inference smoke test

Runs a single greedy generation against a held-out example to verify that the adapter produces correctly cited Turkish legal text.

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(trainer.model)   # Unsloth 2x faster inference

test_q = 'Cumhurbaskanina hakaret eden bir kisinin sosyal medyada paylasmasi durumunda ne olur?'
test_ctx = """[TCK m.299]
(1) Cumhurbaskanina hakaret eden kisi, bir yildan dort yila kadar hapis cezasi ile cezalandirilir.
(2) Sucun alenen islenmesi halinde, verilecek ceza altida biri oraninda artirilir."""

prompt = tokenizer.apply_chat_template([
    {'role': 'system', 'content': SYSTEM},
    {'role': 'user', 'content': f'Soru: {test_q}\n\nKaynak Madde:\n{test_ctx}\n\nCevap:'},
], tokenize=False, add_generation_prompt=True)

ins = tokenizer(prompt, return_tensors='pt').to('cuda')
out = trainer.model.generate(
    **ins,
    max_new_tokens=200,
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
)
print(tokenizer.decode(out[0][ins['input_ids'].shape[1]:], skip_special_tokens=True))

## Export

The LoRA adapter (~200 MB) is downloaded from the Kaggle Output panel and placed at `data/models/llm_adapter/` locally. At inference time it is attached on top of the same base model:

```python
from unsloth import FastLanguageModel
sft_model, tok = FastLanguageModel.from_pretrained(
    'Trendyol/Trendyol-LLM-7B-chat-v4.1.0',
    max_seq_length=2048, load_in_4bit=True,
)
sft_model.load_adapter('data/models/llm_adapter')
```

The base-vs-SFT comparison appears as the `A5 / A5a` pair in the 8-cell ablation matrix (see `04_ablation_eval.ipynb`).